# FactOwl

## 0. Setting up imports and environment variables

In [1]:
# %load_ext autoreload
# %autoreload 2
# %cd /my_dir/factowl/
# !pip install -e ./
# %cd factowl/

In [2]:
!python --version

Python 3.10.13


In [3]:
!ls /my_dir/factowl

LICENSE      cli.py	    factowl	      requirements.txt
README.md    data	    factowl.egg-info  run.py
__init__.py  example.ipynb  factowl.yml       run.sh
__main__.py  examples	    pyproject.toml    setup.py


In [4]:
# !pip -q install wikipedia

In [5]:
%cd /my_dir/factowl

/my_dir/factowl


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [6]:
!git status .

fatal: detected dubious ownership in repository at '/my_dir/factowl'
To add an exception for this directory, call:

	git config --global --add safe.directory /my_dir/factowl


In [7]:
!pwd

/my_dir/factowl


In [8]:
!nvidia-smi

Thu Mar 19 12:10:20 2026       
+---------------------------------------------------------------------------------------+
| NVIDIA-SMI 535.288.01             Driver Version: 535.288.01   CUDA Version: 12.2     |
|-----------------------------------------+----------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |         Memory-Usage | GPU-Util  Compute M. |
|                                         |                      |               MIG M. |
|=========================================+======================+======================|
|   0  NVIDIA GeForce RTX 3090        Off | 00000000:41:00.0 Off |                  N/A |
|  0%   25C    P8              26W / 370W |      0MiB / 24576MiB |      0%      Default |
|                                         |                      |                  N/A |
+-----------------------------------------+----------------------+--

In [9]:
%cd /my_dir/factowl/factowl/
!pip install -e ./

/my_dir/factowl/factowl
Obtaining file:///my_dir/factowl/factowl
ERROR: file:///my_dir/factowl/factowl does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


In [10]:
import argparse
import json
import logging
import numpy as np
import os
import pandas as pd

from huggingface_hub import login
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

from factowl.factscorer import FactowlFactScorer as FactScorer
from factowl.io import save_predictions, save_eval_results, load_simple_json, load_json_generations
from vllm import LLM, SamplingParams
import os

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
/opt/conda/lib/python3.10/site-packages/jieba/_compat.py:18: UserWarning: Module factowl was already imported from None, but /my_dir/factowl/factowl is being added to sys.path
  import pkg_resources


In [11]:
%cd /my_dir/factowl_evaluation/factscore_data

/my_dir/factowl_evaluation/factscore_data


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


## 1. Setting up the parameters for generation

In here, `hf_token` is needed to load the model from HuggingFace, `model_name` is the model's name. Point `data_dir` to the directory, where the downloaded wikipedia dump is located. Your data, which will be evaluated, should be placed in `file_path`. Additionally, create a cache directory and point towards it using `cache_dir` variable.

In [12]:
!ls /my_dir/wikipedia_dumps/factscore_data/data

In [13]:
hf_token = 'hf_token'
model_name = 'meta-llama/Meta-Llama-3-8B-Instruct'
cache_dir = './cachedir/'
data_dir = '/my_dir/wikipedia_dumps/'
file_path = '/my_dir/wikipedia_dumps/data/labeled'
cnp = 1 # Number of context pages retrieved from Wikipedia API.
nsp = 5 # Number of relevant passages retrieved to support a single atomic fact.
# Retrieval source: 'db' for local Wikipedia dump or 'wikipedia_api' for API search
context_type = 'wikipedia_api'
# Set to true for biography-specific postprocessing on FactScore data
is_bio = False

## 2. Initialize VLLM engine for generation

We use VLLM to increase the efficiency of our factchecking engine.

In [14]:
os.environ["CUDA_VISIBLE_DEVICES"]="0"
os.environ["VLLM_LOG_LEVEL"] = "WARNING"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

vllm_model = LLM(
    model=model_name,
    gpu_memory_utilization=0.8,
    dtype="half",
    trust_remote_code=True,
)

INFO 03-19 12:10:31 [utils.py:261] non-default args: {'trust_remote_code': True, 'dtype': 'half', 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'meta-llama/Meta-Llama-3-8B-Instruct'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.
The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


INFO 03-19 12:10:32 [model.py:541] Resolved architecture: LlamaForCausalLM
WARNING 03-19 12:10:32 [model.py:1885] Casting torch.bfloat16 to torch.float16.
INFO 03-19 12:10:32 [model.py:1561] Using max model len 8192
INFO 03-19 12:10:32 [scheduler.py:226] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 03-19 12:10:32 [vllm.py:624] Asynchronous scheduling is enabled.
(EngineCore_DP0 pid=7751) INFO 03-19 12:10:34 [core.py:96] Initializing a V1 LLM engine (v0.15.1) with config: model='meta-llama/Meta-Llama-3-8B-Instruct', speculative_config=None, tokenizer='meta-llama/Meta-Llama-3-8B-Instruct', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.float16, max_seq_len=8192, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=

Loading safetensors checkpoint shards:   0% Completed | 0/4 [00:00<?, ?it/s]


(EngineCore_DP0 pid=7751) INFO 03-19 12:10:45 [default_loader.py:291] Loading weights took 5.39 seconds
(EngineCore_DP0 pid=7751) INFO 03-19 12:10:45 [gpu_model_runner.py:4130] Model loading took 14.96 GiB memory and 6.989975 seconds
(EngineCore_DP0 pid=7751) INFO 03-19 12:10:51 [backends.py:812] Using cache directory: /root/.cache/vllm/torch_compile_cache/eca23dd91d/rank_0_0/backbone for vLLM's torch.compile
(EngineCore_DP0 pid=7751) INFO 03-19 12:10:51 [backends.py:872] Dynamo bytecode transform time: 5.74 s
(EngineCore_DP0 pid=7751) INFO 03-19 12:10:56 [backends.py:267] Directly load the compiled graph(s) for compile range (1, 8192) from the cache, took 1.817 s
(EngineCore_DP0 pid=7751) INFO 03-19 12:10:56 [monitor.py:34] torch.compile takes 7.55 s in total
(EngineCore_DP0 pid=7751) INFO 03-19 12:10:58 [gpu_worker.py:356] Available KV cache memory: 2.72 GiB
(EngineCore_DP0 pid=7751) INFO 03-19 12:10:58 [kv_cache_utils.py:1307] GPU KV cache size: 22,288 tokens
(EngineCore_DP0 pid=775

Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:04<00:00, 12.32it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 35/35 [00:02<00:00, 16.94it/s]


(EngineCore_DP0 pid=7751) INFO 03-19 12:11:05 [gpu_model_runner.py:5063] Graph capturing finished in 7 secs, took 2.07 GiB
(EngineCore_DP0 pid=7751) INFO 03-19 12:11:05 [core.py:272] init engine (profile, create kv cache, warmup model) took 19.59 seconds
INFO 03-19 12:11:07 [llm.py:343] Supported tasks: ['generate']


## 3. Set up the names of the target files and evaluation setup.

In here, you can set up the names of the files to check and determine, whether you want to use npm for checking or not.

In [15]:
import logging
# logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s',
#                         datefmt='%Y-%m-%d %H:%M:%S', )

DEBUG=False
if DEBUG:
    logging.basicConfig(level=logging.DEBUG)

## 4. Evaluation on FactScore data 

In [16]:
model_name = 'meta-llama/Meta-Llama-3-8B-Instruct'
eval_dict = {}

gen_name2abstain = {
    "ChatGPT": "generic",
    "InstructGPT": "generic",
    "PerplexityAI": "perplexity_ai"
}
gen_name2setup = {
    "ChatGPT": "retrieval+llama",
    "InstructGPT": "retrieval+llama",
    "PerplexityAI": "retrieval+llama"
}

gen_names = ["ChatGPT", "InstructGPT", "PerplexityAI"]
eval_dict = {}


for j, gn in enumerate(gen_names):
    st = gen_name2setup[gn]
    print(f"Evaluating {gn}")
    print("model_name", model_name)

    json_p = os.path.join(file_path, f"{gn}.jsonl")
    abstain_type = gen_name2abstain[gn]
    atomic_facts_cache_dir = f"./cache/{gn}-{st}-{context_type}-p{cnp}-c{nsp}/"
    print(f"{atomic_facts_cache_dir=}")
    
    fs = FactScorer(model_name=st,
            data_dir=data_dir,
            vllm_model=vllm_model,
            verifier_max_tokens=16,
            fact_generator_max_tokens=4096,
            dump_every_int=200,
            cache_dir=cache_dir,
            abstain_detection_type=abstain_type,
            is_bio=True,
            retrieval_device="cuda:0",
            context_type=context_type,
            context_num_pages=cnp,
            num_supporting_contexts=nsp,
            filter_facts=False,
            debug=DEBUG)

    topics, generations = load_json_generations(json_p)
    # topics, generations = topics[:10], generations[:10]
    # This will produce to files: predictions and scores
    save_p=f"/my_dir/evaluation/eval_results/factscore_dataset/facts_{gn}-{st}_{context_type}-p{cnp}-c{nsp}.tsv"


    out = fs.get_score(topics, generations, save_path=save_p, gamma=10, knowledge_source="enwiki-20230401", verbose=True)
    eval_dict[gn] = out

    print(f'Score: {out["score"]}\nRespond ratio: {out["respond_ratio"]}')

[2026-03-19 12:11:07] INFO factscorer.py:84: FactScore is using context retrieval type: wikipedia_api


Evaluating ChatGPT
model_name meta-llama/Meta-Llama-3-8B-Instruct
atomic_facts_cache_dir='./cache/ChatGPT-retrieval+llama-wikipedia_api-p1-c5/'


100%|██████████| 157/157 [00:00<00:00, 296757.88it/s]

Starting fact generation


Adding requests:   0%|          | 0/434 [00:00<?, ?it/s]

...

Adding requests:   0%|          | 0/479 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/479 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Fact generation took 45.40761709213257 seconds


100%|██████████| 158/158 [11:37<00:00,  4.41s/it]

Saving atomic facts DataFrame. Size: (3795, 7), Columns: Index(['sample_id', 'topic', 'atom', 'is_supported', 'label', 'context',
       'num_context_passages'],
      dtype='object')
Score: 0.6597719314885947
Respond ratio: 1.0


# Evaluation on your data

In [20]:

base_dir = "./"
atomic_facts_cache_dir = os.path.join(base_dir, "cache/")
save_dir = os.path.join(base_dir, "results/")

predictions_path = os.path.join(save_dir, "predictions.tsv")
scores_path = os.path.join(save_dir, "scores.tsv")

# Load Free-form generations to be evaluated and generation topics
topics, generations = <LOAD_YOUR_DATA_HERE>
# topics, generations = ["William Post"], ["This is a test sentence on William Post.", ]


fs = FactScorer(model_name=st,
        data_dir=data_dir,
        vllm_model=vllm_model,
        verifier_max_tokens=16,
        fact_generator_max_tokens=4096,
        dump_every_int=200,
        cache_dir=cache_dir,
        abstain_detection_type="generic",
        is_bio=False,
        retrieval_device="cuda:0",
        context_type=context_type,
        context_num_pages=cnp,
        num_supporting_contexts=nsp,
        filter_facts=True,
        debug=DEBUG)


save_p=<SET_PATH_TO_FACTS_TSV>
out = fs.get_score(topics, generations, save_path=save_p, gamma=10, knowledge_source="enwiki-20230401", verbose=True)

print(f'Score: {out["score"]}\nRespond ratio: {out["respond_ratio"]}')
